# A first look at the EEG

Three small exercises: a synthetic signal and its spectrum, one real run from the dataset, and leakage made on purpose.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, sosfiltfilt, welch

## 1. Signals, spectra and filters

A 10 Hz rhythm (like mu) plus 50 Hz mains hum plus noise. The 5–35 Hz band-pass is the one the dataset authors applied.

In [ ]:
rng = np.random.default_rng(0)
sfreq = 250.0
t = np.arange(0, 4, 1 / sfreq)
x = (
    np.sin(2 * np.pi * 10 * t)
    + 0.5 * np.sin(2 * np.pi * 50 * t)
    + 0.3 * rng.standard_normal(t.size)
)

f, pxx = welch(x, fs=sfreq, nperseg=500)
sos = butter(4, [5, 35], btype="bandpass", fs=sfreq, output="sos")
y = sosfiltfilt(sos, x)
fy, pyy = welch(y, fs=sfreq, nperseg=500)

fig, ax = plt.subplots(2, 1, figsize=(8, 6))
ax[0].plot(t[:250], x[:250], label="raw")
ax[0].plot(t[:250], y[:250], label="5-35 Hz")
ax[0].set_xlabel("s")
ax[0].legend()
ax[1].semilogy(f, pxx, label="raw")
ax[1].semilogy(fy, pyy, label="filtered")
ax[1].set_xlabel("Hz")
ax[1].legend()
plt.tight_layout()

In [ ]:
def power_at(freqs, power, hz):
    return power[np.argmin(np.abs(freqs - hz))]


print(f"10 Hz kept to   {power_at(fy, pyy, 10) / power_at(f, pxx, 10):.2f} of its power")
print(f"50 Hz reduced to {power_at(fy, pyy, 50) / power_at(f, pxx, 50):.1e} of its power")

## 2. One real run

`sub-01`, pre-training session, run 1. Codes in the annotations: 1, 2 and 3 are the trial events (see concept note 10); `BAD_boundary` marks where the authors cut data out.

In [ ]:
from sixthfinger.data.loading import load_processed_run
from sixthfinger.paths import find_root

run_path = find_root() / "data/processed/sub-01/pre/sub-01_B_pre_01_eeg.mat"
raw, info = load_processed_run(run_path)
print(info)
print(raw)
print(f"{len(raw.ch_names)} channels, {raw.info['sfreq']:.0f} Hz, {raw.times[-1]:.0f} s")

In [ ]:
raw.plot(duration=20, n_channels=10, scalings=dict(eeg=20e-6));

In [ ]:
raw.compute_psd(fmax=40).plot();

In [ ]:
ann = raw.annotations
print(ann[:8])
onsets = {code: ann.onset[ann.description == code] for code in sorted(set(ann.description))}
for code, times in onsets.items():
    gap = np.median(np.diff(times)) if len(times) > 1 else float("nan")
    print(f"{code:>14}: {len(times):3d} events, median gap {gap:.1f} s")

In [ ]:
rest = onsets.get("1", np.array([]))
cues = np.concatenate([onsets.get(c, np.array([])) for c in ("2", "3")])
gaps = [cues[cues > r].min() - r for r in rest if (cues > r).any()]
print(f"rest marker to next cue: median {np.median(gaps):.1f} s")

## 3. Leakage with your own hands

Pure noise, random labels. Choosing features on all the data before cross-validation lets the test folds shape the model, the same mistake as fitting CSP on all trials before splitting.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

leaky, honest = [], []
for seed in range(20):
    noise = np.random.default_rng(seed)
    X = noise.standard_normal((80, 2000))
    y = noise.integers(0, 2, 80)
    X_sel = SelectKBest(f_classif, k=20).fit_transform(X, y)
    leaky.append(cross_val_score(LogisticRegression(), X_sel, y, cv=10).mean())
    pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
    honest.append(cross_val_score(pipe, X, y, cv=10).mean())

# One draw of 80 trials is noisy (an honest 0.35 to 0.7 is normal), so average 20.
print(f"leaky : {np.mean(leaky):.2f} (min {np.min(leaky):.2f})")
print(f"honest: {np.mean(honest):.2f} (range {np.min(honest):.2f}-{np.max(honest):.2f})")